In [ ]:
%load_ext autoreload
%autoreload 2
from kg.cleaning.referencing import PipelineBuilder, EntityMapper
from PyPDF2 import PdfReader
import os
import amrlib
import penman
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch
from amrlib import load_stog_model
import networkx as nx
import sys, os
from pyvis.network import Network
import os
import json
import xml.etree.ElementTree as ET
import psutil
import os
import signal

import epo_ops
import spacy
from dotenv import load_dotenv
from tools.sentence.entity import Entity, InMemoryEntityRepository

from PatentProvider import PatentProvider
import random
import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json
import json
import random
import os
import spacy
from spacy.tokens import DocBin
import torch
import matplotlib.pyplot as plt
import amrlib
import spacy
from tools.sentence.sentence import Sentence
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from kg.formatting.formatting_manager import FormattingManager
import sys, importlib,os
sys.modules.pop('PatentTextFormatter', None)
importlib.invalidate_caches()
import numpy as np
from kg.cleaning.normalising.word_normaliser import WordNormaliser
from fastcoref import FCoref
# Import extensions to register spaCy components
from kg.cleaning.referencing import extensions  # noqa: F401
from tools.sentence.entity import Entity
import torch, os, platform
from kg.generating_kg.generating.NodeGenerator import NodeGenerator
import spacy
import os, pyvis, jinja2, sys
import epo_ops
import epo_ops
import os
import spacy
import xml.etree.ElementTree as ET
import json
import sys
from kg.generating_kg.analysing.TextEncoder import TextEncoder
from PatentProvider import PatentProvider
# Import new graph processing classes
from tools.graph.visualizer import GraphVisualizer
from tools.graph.faiss_merger import FAISSEdgeMerger
from tools.graph.cluster_manager import ClusterManager
from tools.graph.neo4j_manager import Neo4jManager
from kg.formatting.formatting_manager import FormattingManager
from tools.sentence.sentence import Sentence
from tools.sentence.sentence_classifier import SentenceClassifier
from dataclasses import dataclass
from typing import List
from kg.generating_kg.generating.ParallelTripleGenerator import ParallelTripleGenerator
from tools.graph.relation_decomposer import RelationDecomposer
import networkx as nx
import logging
import importlib
import web_editor.graph_validator_chat
#from web_editor.graph_validator_chat import start_validator_chat
from tools.graph.visualizer import GraphVisualizer
from tools.graph.kg_gen_converter import build_id_to_name_map
# Initialize decomposer
from tools.graph.relation_simplifier import RelationSimplifier
from web_editor.graph_validator_chat import start_validator_chat


c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
01/11/2026 17:14:02 - INFO - 	 Loading faiss with AVX512 support.
01/11/2026 17:14:02 - INFO - 	 Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
01/11/2026 17:14:02 - INFO - 	 Loading faiss with AVX2 support.
01/11/2026 17:14:02 - INFO - 	 Successfully loaded faiss with AVX2 support.


In [2]:
import torch

if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.7)  # 50% of VRAM

    torch.set_num_threads(4)
    torch.set_num_interop_threads(1)


In [ ]:
nlp = spacy.load("en_core_web_trf")   # see optimization ideas below
formatterManager = FormattingManager()
random_description = PatentProvider().getDescription("1502502")

from kg.formatting.formatting_manager import FormattingManager

# Initialize the formatting manager
# Default: 8 workers for retrieveContent, 12 for split

# Custom workers
fm = FormattingManager(num_workers=8, split_workers=12)
# Extract only invention-related sentences
sentences = fm.retrieveContent(random_description, chunk_size=1000)

# Returns a list of Sentence objects
for sentence in sentences:
    print(sentence.text)

In [ ]:

fm = FormattingManager()

# Returns List[Sentence], not List[str]
split_sentences = fm.split(sentences)




In [ ]:

# Initialize classifier (uses GPU if available, batch processing enabled)
classifier = SentenceClassifier(
    model_path="training/info/done/hf/sentence_classifier_model",
    batch_size=32,  # Adjust based on GPU memory
    use_gpu=True
)

# Filter sentences to keep only informative ones (much faster with batch processing)
sentence_split = classifier.filter_informative(
    split_sentences, 
    keep_labels=["INFORMATIVE"]  # Can also include ["INFORMATIVE", "FIGURE_RELATED"] if needed
)



In [ ]:

@dataclass(frozen=True)
class JoinedText:
    """
    Holds the concatenated text passed to spaCy
    and the starting character offset of each sentence.
    """
    text: str
    starts: List[int]


def join_sentences(sentences, sep=" "):
    """
    Join a list of Sentence objects into one string while
    tracking sentence start offsets.

    Args:
        sentences: List of Sentence objects with `.text`
        sep: Separator inserted between sentences (default: space)

    Returns:
        JoinedText(text, starts)
    """
    parts = []
    starts = []
    cur = 0

    for i, s in enumerate(sentences):
        starts.append(cur)
        parts.append(s.text)
        cur += len(s.text)

        if i < len(sentences) - 1:
            parts.append(sep)
            cur += len(sep)

    return JoinedText("".join(parts), starts)


In [ ]:
pipeline_builder = PipelineBuilder()
entity_mapper = EntityMapper(sentence_cls=Sentence)

joined = join_sentences(sentence_split, sep=" ")
doc = pipeline_builder.nlp(joined.text)

clusters = entity_mapper.map_to_sentences(doc, sentence_split, joined)

print("doc.ents:", len(doc.ents))
print("coref clusters:", len(doc._.coref_clusters))
print("entities in first sentence:", len(sentence_split[0].entities))
print(type(sentence_split[0].entities[0]))
print(sentence_split[0].entities[0])

# Collect all entities created by the mapper
all_entities: list[Entity] = []
for sentence in sentence_split:
    all_entities.extend(sentence.entities)

# Create repository and add entities
repo = InMemoryEntityRepository()
for entity in all_entities:
    repo.save(entity)

print("=== REPO CONTENTS ===")
for e in repo.getAll().values():
    print(
        f"Entity("
        f"name={e.name}, "
        f"label={e.label}, "
        f"id={e.id}, "
        f"ref={e.ref}"
        f")"
    )
for ent in doc.ents[:50]:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

# For each sentence, count overlaps
offset = 0
for i, s in enumerate(sentence_split):
    start = offset
    end = start + len(s.text)
    overlaps = [ent for ent in doc.ents if ent.start_char < end and ent.end_char > start]
    print("sentence", i, "overlaps", len(overlaps))
    offset = end + 1



In [ ]:
print(sentence_split)  

In [ ]:

# Initialize parallel triple generator
# - 10 workers for parallel processing
# - Rate limit: 900 calls/minute (stays below 1000 limit)
generator = ParallelTripleGenerator(
    repo=repo,max_workers=10,
    rate_limit_per_minute=900,
    verbose=True
)

# Generate triples from sentences (handles all parallelization, rate limiting, etc.)
triples = generator.generate(sentence_split)


In [ ]:
print(triples)

In [ ]:
# --- Merge relations per (head, tail) using FAISSEdgeMerger
# Initialize merger
merger = FAISSEdgeMerger(
    sim_threshold=0.8,
    embed_dim=256,
    ngram=3,
    keep="shortest",
)

# Merge relations
triples, merge_stats = merger.merge_relations(triples)

print("Merge stats:", merge_stats)
print("Before:", len(triples), "After:", len(triples))



In [ ]:


# Use the simplifier (Option 2 - RECOMMENDED)
simplifier = RelationSimplifier(
    max_relation_length=4,
    verbose=True
)

# Simplify triples (keeps same structure, adds properties)
triples = simplifier.simplify(triples)

In [ ]:
# --- Build + visualize a typed KG from List[Triple] using GraphVisualizer

# Initialize visualizer
visualizer = GraphVisualizer()

# Build ID -> Name map from sentence_split
id_to_name = visualizer.build_id_to_name_map(sentence_split)

# Build graph from triples
G = visualizer.build_graph(triples)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Visualize
visualizer.visualize_pyvis(G, out_file="graph_merged.html", id_to_name=id_to_name)


In [ ]:
# Save variables
%store triples
%store G
%store sentence_split

In [2]:
# 
# Or reload all stored variables at once
%store -r

In [ ]:

logging.basicConfig(
    level=logging.INFO,  # Use DEBUG for more detail
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

<module 'web_editor.graph_validator_chat.server' from 'c:\\Users\\Caleb\\Documents\\LLM Patent Claim Generator Thesis\\web_editor\\graph_validator_chat\\server.py'>

[API] GET /api/status
[127.0.0.1] "GET /api/status HTTP/1.1" 200 -
[API] GET /api/state
[127.0.0.1] "GET /api/state HTTP/1.1" 200 -
[API] GET /api/triples
[127.0.0.1] "GET /api/triples HTTP/1.1" 200 -


Killing PID 998760


In [3]:
import os
os.system("taskkill /IM node.exe /F")


0

In [3]:
import importlib
import web_editor.graph_validator_chat.server
importlib.reload(web_editor.graph_validator_chat.server)

PORT = 3000

killed_pids = set()

for conn in psutil.net_connections(kind="inet"):
    if conn.laddr and conn.laddr.port == PORT and conn.status == psutil.CONN_LISTEN:
        pid = conn.pid
        if pid and pid not in killed_pids:
            print(f"Killing PID {pid}")
            os.kill(pid, signal.SIGTERM)
            killed_pids.add(pid)
        
%store -r
# Fix Jinja2 compatibility - run this FIRST
import jinja2
if not hasattr(jinja2, ''):
    from markupsafe import escape
    jinja2.escape = escape
    import logging
import sys
import logging
# Configure logging to output to stdout (visible in Jupyter cells)
logging.basicConfig(
    level=logging.INFO,  # or logging.DEBUG for more detail
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,  # This ensures output goes to the cell
    force=True  # Override any existing c
)

# Now your logging will appear in the cell
# OP
# Now your normal imports will work
from tools.graph.kg_gen_converter import build_id_to_name_map
# Build id_to_name mapping
id_to_name = build_id_to_name_map(triples)

# Start chat (browser opens automatically)
# Use LangGraph validator (default)

# Enable debug mode
start_validator_chat(
    graph=G,
    triples=triples,
    id_to_name=id_to_name,
    debug=True,
    port = 50025  # <-- Enable debug mode
)

Killing PID 964996
[Server] Requested API port: 50025
✓ Using requested port 50025 for API server
✓ API Server running on http://localhost:50025
✓ Using port 3000 for Next.js frontend
🚀 Starting Next.js dev server on port 3000...
✓ Graph Validator Chat: http://localhost:3000
✓ API Server: http://localhost:50025
[Server] Initializing validator in background...

[Server] ✓ Servers are running. This cell will stay ACTIVE.
[Server] ✓ Debugger will remain attached while this loop runs.
[Server] ✓ Press Ctrl+C in this cell to stop servers.
[Server] Entering blocking loop (checking every 0.5s)...
[Server] Current time: 17:18:22
[Server] About to enter loop.
[Server] _server_running = True
[Server] api thread alive = True
[Server] nextjs thread alive = True
[Server] Starting background analysis...
2026-01-11 17:18:23 - httpx - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
[Server] Opening browser to http://localhost:3000

[Server] ⚠️  KeyboardInterrup

2026-01-11 17:18:30 - httpx - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


In [ ]:
from web_editor.graph_validator_chat.helper import get_all_updates

# Get everything
updates = get_all_updates()

# Extract updated data
G_updated = updates['graph']
triples_updated = updates['triples']
entities_updated = updates['entities']
id_to_name_updated = updates['id_to_name']
changes = updates['changes']

print(f"✅ Graph: {G_updated.number_of_nodes()} nodes")
print(f"✅ Triples: {len(triples_updated)} triples")
print(f"✅ Changes: {changes}")


# Visualize the updated graph
if G_updated:
    # Option 1: Visualize the graph directly (if it's already a NetworkX graph)
    print("\n📊 Visualizing updated graph...")
    visualize_nx_browser_full(G_updated, path="updated_graph.html", id_to_name=id_to_name_updated)
elif triples_updated:
    # Option 2: Build graph from triples and visualize
    print("\n📊 Building graph from updated triples and visualizing...")
    visualizer = GraphVisualizer()
    G_from_triples = visualizer.build_graph(triples_updated, deduplicate=True)
    visualize_nx_browser_full(G_from_triples, path="updated_graph.html", id_to_name=id_to_name_updated)
else:
    print("⚠️ No graph or triples to visualize")

In [ ]:
# Reload the module to get the latest changes
import importlib
import sys

# Remove the module from cache
if 'tools.graph.claim_drafting_agent' in sys.modules:
    del sys.modules['tools.graph.claim_drafting_agent']
if 'tools.graph.graph_rag' in sys.modules:
    del sys.modules['tools.graph.graph_rag']
if 'tools.graph' in sys.modules:
    del sys.modules['tools.graph']

# Re-import
from tools.graph import GraphRAG, ClaimDraftingAgent, ClaimExtractor

In [ ]:
from tools.graph import GraphRAG, ClaimDraftingAgent, ClaimExtractor

# Initialize GraphRAG with your graph
graph_rag = GraphRAG(
    G=G_updated,  # Your NetworkX graph
    triples=triples_updated,  # Your triples
    id_to_name=id_to_name_updated,  # Use updated mapping if available, otherwise id_to_name
)

# Initialize ClaimDraftingAgent with RAG
from tools.graph import (
    ClaimDraftingAgent, 
    IndependentClaimProposal, 
    DependentClaimProposal
)

# Create proposals
independent_proposals = [
    IndependentClaimProposal(
        focus="water tank mechanism",
        focus_keywords=["tank", "water", "storage", "convection"],
        focus_categories=["INVENTIVE_MECHANISM"]
    ),
    IndependentClaimProposal(
        focus="bubble generation system",
        focus_keywords=["bubble", "air", "generator", "aeration"],
        focus_entities=["air bubble generating member"]
    )
]

dependent_proposals = [
    # Dependents for first independent (index 0)
    DependentClaimProposal(
        parent_independent_index=0,
        focus="opening portion configuration",
        focus_keywords=["opening", "portion", "convection"]
    ),
    DependentClaimProposal(
        parent_independent_index=0,
        focus="conduit arrangement",
        focus_keywords=["conduit", "pipe", "connection"]
    ),
    DependentClaimProposal(
        parent_independent_index=0,
        focus="storage positioning",
        focus_keywords=["storage", "elevated", "position"]
    ),
    # Dependents for second independent (index 1)
    DependentClaimProposal(
        parent_independent_index=1,
        focus="bubble generator types",
        focus_keywords=["perforated", "porous", "tube"]
    ),
    DependentClaimProposal(
        parent_independent_index=1,
        focus="air flow control",
        focus_keywords=["air", "flow", "control"]
    )
]

# Use with drafting agent
drafting_agent = ClaimDraftingAgent(graph_rag=graph_rag)
drafted_claims = drafting_agent.draft(
    G=G,
    independent_proposals=independent_proposals,
    dependent_proposals=dependent_proposals,
    patent_description=text_input
)


In [ ]:
from tools.graph import print_claims
print_claims(drafted_claims)

In [ ]:
# --- Filter relations using LLM to remove unnecessary, duplicate, or uninformative relations
from tools.graph.llm_relation_filter import LLMRelationFilter

# Initialize filter
# review_mode options: "strict" (most aggressive), "moderate" (balanced), "lenient" (conservative)
relation_filter = LLMRelationFilter(
    review_mode="moderate",  # Adjust based on how aggressive you want filtering
    batch_size=10,  # Number of nodes to process (not used in current implementation, but kept for future)
)

# Filter relations
# This will review relations for each node and remove unnecessary/duplicate/uninformative ones
filtered_triples, filter_stats = relation_filter.filter_relations(
    triples=merged_triples,  # Use merged_triples from FAISS merger
    id_to_name=id_to_name,  # Optional: provides better entity names for LLM context
)

print("\nFilter stats:", filter_stats)

# Update triples
triples = filtered_triples

# Visualize filtered graph
G_filtered = visualizer.build_graph(filtered_triples)
print("\nFiltered Nodes:", G_filtered.number_of_nodes())
print("Filtered Edges:", G_filtered.number_of_edges())
visualizer.visualize_pyvis(G_filtered, out_file="graph_filtered.html", id_to_name=id_to_name)


In [ ]:
print(triples)
print(random_description)

In [ ]:
# --- Rule-driven seed clusters using ClusterManager
# Build graph from merged_triples if available, else triples
input_triples = merged_triples if "merged_triples" in globals() else triples
print("Using:", "merged_triples" if "merged_triples" in globals() else "triples")

# Build graph
G = visualizer.build_graph(input_triples)

# Initialize cluster manager with default rules
cluster_manager = ClusterManager(G)

# Create rule-based clusters
clusters = cluster_manager.create_rule_based_clusters()

# Assign edges to clusters
cid_to_seedtype = cluster_manager.assign_edges_to_clusters(clusters)

# Add colors to edges
from tools.graph.visualizer import EDGE_CLUSTER_COLORS, EDGE_COLOR_DEFAULT
for u, v, k, d in G.edges(keys=True, data=True):
    cide = d.get("cluster_id", -1)
    d["color"] = EDGE_CLUSTER_COLORS[cide % len(EDGE_CLUSTER_COLORS)] if cide != -1 else EDGE_COLOR_DEFAULT

# Visualize
visualizer.visualize_pyvis(
    G,
    out_file="graph_merged.html",
    id_to_name=id_to_name,
    cluster_attr="cluster_id",
    cid_to_seedtype=cid_to_seedtype,
)


In [ ]:
from web_editor import start_triple_editor, get_updated_triples
   
   # Start the editor with your triples
start_triple_editor(triples, port=5000)
   
   # The browser opens automatically
   # Edit your triples and entities
   # When done, close the browser and run:
updated_triples = get_updated_triples()

In [ ]:
updated_triples = get_updated_triples()

In [ ]:
from tools.graph import print_triples_vertical, print_triples_compact

# Print all triples in vertical format
print_triples_vertical(triples)

# Print first 20 triples
print_triples_vertical(triples, max_triples=20)

# Compact format (one line per triple)
print_triples_compact(triples)

In [ ]:
gemini/gemini-2.5-flash


In [ ]:
from kg_gen import KGGen
import os
from PatentProvider import PatentProvider
# Initialize KGGen with optional configuration
kg = KGGen(
  model="gemini/gemini-2.5-flash",  # Default model
  temperature=0.9,        # Default temperature
  api_key=os.getenv("GOOGLE_API_KEY")  # Optional if set in environment or using a local model
)
print(os.getenv("GOOGLE_API_KEY"))
# EXAMPLE 1: Single string with context
text_input = PatentProvider().getDescription("1502502")
print(text_input)
graph_1 = kg.generate(
  input_data=text_input,
  context="The invention relates to a display device for appreciation that creates a pseudo space (such as underwater or aerial space) in which models of creatures or objects appear to move naturally within a liquid-filled tank."
)
# Output: 
# entities={'Linda', 'Ben', 'Andrew', 'Josh'} 
# edges={'is brother of', 'is father of', 'is mother of'} 
# relations={('Ben', 'is brother of', 'Josh'), 
#           ('Andrew', 'is father of', 'Josh'), 
#           ('Linda', 'is mother of', 'Josh')}

In [ ]:
# Remove singular nodes from kg_gen graph
def remove_singular_nodes_from_kg_gen(kg_graph):
    entities = getattr(kg_graph, "entities", set())
    relations = getattr(kg_graph, "relations", set())
    
    entities_in_relations = set()
    for relation in relations:
        if isinstance(relation, tuple) and len(relation) >= 3:
            entities_in_relations.add(relation[0])  # subject
            entities_in_relations.add(relation[2])  # object
        elif hasattr(relation, "subject") and hasattr(relation, "object"):
            entities_in_relations.add(relation.subject)
            entities_in_relations.add(relation.object)
    
    singular_entities = entities - entities_in_relations
    
    if hasattr(kg_graph, "entities"):
        if isinstance(kg_graph.entities, set):
            kg_graph.entities -= singular_entities
        elif isinstance(kg_graph.entities, dict):
            for entity in singular_entities:
                kg_graph.entities.pop(entity, None)
    
    print(f"✅ Removed {len(singular_entities)} singular node(s)")
    return kg_graph

# Apply to your graph


In [ ]:
from tools.graph.visualizer import GraphVisualizer
from tools.graph.assertion_agent import AssertionAgent
from tools.graph.claim_concept_agent import ClaimConceptAgent
from tools.graph.claim_extractor import ClaimExtractor
from tools.graph.claim_drafting_agent import ClaimDraftingAgent
# Convert kg_gen graph to triples
from tools.graph.kg_gen_converter import kg_gen_graph_to_triples
from tools.graph.visualize_helper import visualize_nx_browser_full
# Convert graph_1 to triples
triples = kg_gen_graph_to_triples(graph_1)

# Force reload modules to get latest code
import importlib
import tools.graph.visualizer
import tools.graph.assertion_agent
importlib.reload(tools.graph.visualizer)
importlib.reload(tools.graph.assertion_agent)
from tools.graph.visualizer import GraphVisualizer
from tools.graph.assertion_agent import AssertionAgent
graph_1 = remove_singular_nodes_from_kg_gen(graph_1)
# Start with your KG-gen graph (from triples)
visualizer = GraphVisualizer()
G = visualizer.build_graph(triples, deduplicate=True)
visualizer.remove_singular_nodes(G)  # Removes all nodes with no edges
# Step 1: Add assertions
inventive_desc = "A water tank for appreciation provided with a water storage, a water pipe, an air bubble generating member, wherein said water storage is provided with a water inlet port and an opening portion and said opening portion is so provided that the convection can be generated in said water tank for appreciation by the liquid current from a water tank through said opening portion, and said water storage is installed upwardly of a water tank for appreciation and one end of said water pipe is connected to an inlet water port of a water storage and the other end is so installed that it is in the liquid when the liquid is filled in a water tank for appreciation and an air bubble generating member is so installed that it can lead the air bubble inside of a water pipe in the peripheral portion of said other end of said water pipe and a display device for appreciation using said water tank for appreciation are used."

assertion_agent = AssertionAgent(inventive_description=inventive_desc)
G = assertion_agent.run(G, triples=triples)
visualize_nx_browser_full(G)

# Step 2: Create claim concepts
claim_concept_agent = ClaimConceptAgent()
# Uses claim_eligible=True filter (all claim-eligible assertions, regardless of status)
G = claim_concept_agent.run(G, num_independent=3, num_dependent_per_independent=4)
visualize_nx_browser_full(G)

# Build id_to_name mapping from triples
from tools.graph.kg_gen_converter import build_id_to_name_map

id_to_name = build_id_to_name_map(triples)
# Step 3: Extract claim bundles
extractor = ClaimExtractor(id_to_name=id_to_name)
claim_bundles = extractor.extract(G)

# Step 4: Draft claims
drafting_agent = ClaimDraftingAgent()
# Pass patent description for context (helps LLM understand the invention better)
drafted_claims = drafting_agent.draft(claim_bundles, patent_description=text_input)

# Print numbered claims
for claim in drafted_claims:
    print(f"{claim.claim_number}. {claim.claim_text}")

In [ ]:
print(graph_1.entities)
print(graph_1.edges)
print(graph_1.relations)

In [ ]:
KGGen.visualize(graph_1, "output_path.html", open_in_browser=True)